In [0]:
from pyspark.sql import functions as F

In [0]:
df = (
    spark.range(1, 10_000_000)
    .withColumn("value", F.rand() * 1000)
    .withColumn("id",( F.rand() * 10_000_000).cast("int"))
    .select("id", "value")
)

df.display()

In [0]:
df.write.format("delta").saveAsTable("hive_metastore.rainbow.non_bucketed_table1")

In [0]:
df.write.format("delta").saveAsTable("hive_metastore.rainbow.non_bucketed_table2")

In [0]:
spark.conf.set("spark.databricks.optimizer.adaptive.enabled", "false")

In [0]:
# non-bucketed join

non_bucketed_join = (
    spark.table("hive_metastore.rainbow.non_bucketed_table1")
    .join(spark.table("hive_metastore.rainbow.non_bucketed_table2"), on="id", how="inner")
)
non_bucketed_join.show(truncate=False)

In [0]:
df.orderBy(F.col("id")).write.format("parquet").bucketBy(8, "id").mode("overwrite").saveAsTable("hive_metastore.rainbow.bucketed_table01")

In [0]:
df.orderBy(F.col("id")).write.format("parquet").bucketBy(8, "id").mode("overwrite").saveAsTable("hive_metastore.rainbow.bucketed_table02")

In [0]:
# bucketed join

bucketed_join = (
    spark.table("hive_metastore.rainbow.bucketed_table01")
    .join(spark.table("hive_metastore.rainbow.bucketed_table02"), on="id", how="inner")
)
bucketed_join.show(truncate=False)

In [0]:
%fs
ls dbfs:/user/hive/warehouse/rainbow.db/bucketed_table01